# 2장 머신러닝은 어떻게 배우나 — 데이터, 특징, 레이블, 그리고 일반화
『머신러닝과 딥러닝 기초』 실습 노트북

남극 펭귄 데이터로 다음을 해 봅니다.
1. 데이터를 불러와 **특징**과 **레이블**을 나눈다
2. 정답을 주고 배우는 **지도 학습** — 회귀와 분류
3. **일반화** — 똑같은 것만 찾는 모델과 비슷한 것으로 미루어 답하는 모델의 차이
4. 정답 없이 묶는 **비지도 학습** 맛보기

위에서부터 차례로 실행하세요.

## 1. 데이터 불러오기
파머 펭귄(Palmer Penguins) 데이터는 남극 파머 기지 근처 세 섬에서 펭귄 344마리를 측정한 기록입니다. seaborn 라이브러리로 바로 불러올 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("penguins")
print(df.shape)        # (행 수, 열 수)
df.head()

한 행이 펭귄 한 마리(**샘플**)이고, 한 열이 측정한 항목입니다. `species`는 펭귄의 종입니다.

In [ ]:
df["species"].value_counts()

## 2. 특징과 레이블 나누기
모델에 넣을 입력(**특징**, feature)과 맞혀야 할 정답(**레이블**, label)을 나눕니다. 측정값이 비어 있는 2마리는 뺍니다.

In [ ]:
features = ["bill_length_mm", "bill_depth_mm",
            "flipper_length_mm", "body_mass_g"]
df = df.dropna(subset=features).reset_index(drop=True)

X = df[features]      # 특징: 부리 길이 · 깊이, 날개 길이, 몸무게
y = df["species"]     # 레이블: 종
print("샘플 수:", len(df))
print("특징 수:", X.shape[1])

### 그래프로 보기
부리 길이와 날개 길이만으로도 세 종이 꽤 잘 나뉘어 보입니다.

In [ ]:
sns.scatterplot(data=df, x="bill_length_mm", y="flipper_length_mm", hue="species")
plt.show()

## 3. 회귀: 날개 길이로 몸무게 예측하기
맞힐 값이 종(범주)이 아니라 몸무게(숫자)이면 **회귀** 문제입니다. 1장 택시 요금처럼 직선 규칙을 찾습니다.

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(df[["flipper_length_mm"]], df["body_mass_g"])
w, b = reg.coef_[0], reg.intercept_
print(f"몸무게 ≈ {w:.1f} × 날개 길이 + ({b:,.0f})")

new = pd.DataFrame({"flipper_length_mm": [200]})
print(f"날개 길이 200 mm → 예상 몸무게 {reg.predict(new)[0]:,.0f} g")

## 4. 일반화: 처음 보는 데이터에서도 맞히기
특징 두 개(부리 길이, 날개 길이)만 써서 종을 맞혀 봅니다.

먼저 데이터를 **학습 데이터**(75%)와 **테스트 데이터**(25%)로 나눕니다. 테스트 데이터는 학습이 끝날 때까지 모델에게 보여 주지 않는, 처음 보는 펭귄 역할을 합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X2 = df[["bill_length_mm", "flipper_length_mm"]].values
Xtr, Xte, ytr, yte = train_test_split(
    X2, y.values, test_size=0.25, random_state=42, stratify=y)
print("학습 데이터:", len(Xtr), "마리")
print("테스트 데이터:", len(Xte), "마리")

### 외우는 모델
학습 데이터의 측정값과 정답을 그대로 표에 적어 두었다가, 똑같은 측정값이 나오면 그 정답을 말하는 '모델'을 만들어 봅시다.

In [ ]:
# 측정값 → 정답을 그대로 적어 둔 표
memo = {tuple(x): label for x, label in zip(Xtr, ytr)}

correct = sum(memo[tuple(x)] == t for x, t in zip(Xtr, ytr))
print(f"학습 데이터 정확도: {correct / len(Xtr):.1%}")

answers = [memo.get(tuple(x), "모름") for x in Xte]
answered = sum(a != "모름" for a in answers)
right = sum(a == t for a, t in zip(answers, yte))
print(f"테스트 데이터 {len(Xte)}마리 가운데 답한 수: {answered}")
print(f"답한 것 가운데 맞힌 수: {right}")

학습 데이터는 100% 맞히지만, 처음 보는 펭귄에게는 거의 답하지 못합니다. 측정값이 조금만 달라도 표에 없기 때문입니다.

### 비슷한 것으로 미루어 답하는 모델: k-최근접 이웃
**k-최근접 이웃**(k-nearest neighbors, k-NN)도 학습 데이터를 기억해 둡니다. 다른 점은 처음 보는 펭귄에게 답하는 방식입니다. 새 펭귄과 측정값이 가장 가까운 학습 데이터 k마리를 찾아, 그 가운데 많은 쪽의 종으로 답합니다. 자세한 원리는 4장에서 배웁니다.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [1, 5]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(Xtr, ytr)                         # 학습
    tr, te = model.score(Xtr, ytr), model.score(Xte, yte)
    print(f"k={k}: 학습 정확도 {tr:.1%}, 테스트 정확도 {te:.1%}")

- k=1은 학습 데이터를 100% 맞히지만, 테스트 정확도는 k=5보다 낮습니다(86마리 가운데 2마리 차이 — 나누는 방법에 따라 달라질 수 있으며, 더 믿을 만한 비교 방법은 4장에서 배웁니다).
- 학습 데이터 정확도가 높다고 좋은 모델이 아닙니다. **처음 보는 데이터에서의 정확도**가 진짜 실력입니다.

In [ ]:
model = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
new_penguin = [[45.0, 215.0]]   # 부리 길이 45 mm, 날개 길이 215 mm
print("예측한 종:", model.predict(new_penguin)[0])

## 5. 비지도 학습 맛보기: 정답 없이 묶기
이번에는 종(정답)을 알려 주지 않고, 측정값만 보고 비슷한 펭귄끼리 세 무리로 묶어 봅니다. **k-평균 군집**(k-means clustering)이라는 방법이며 6장에서 자세히 배웁니다. 두 특징은 단위(mm)는 같지만 값이 퍼진 정도가 크게 달라(표준편차 부리 길이 약 5.5 mm, 날개 길이 약 14 mm), 그대로 거리를 재면 날개 길이 차이가 훨씬 크게 반영됩니다. 그래서 먼저 크기를 맞춥니다(**스케일링**, 3장). 앞의 k-최근접 이웃도 거리를 쓰므로 같은 문제가 있으며, 3장에서 다시 확인합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

Z = StandardScaler().fit_transform(X2)    # 특징의 크기 맞추기(3장)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
ct = pd.crosstab(km.labels_, df["species"],
                 rownames=["무리"], colnames=["실제 종"])
print(ct)
print("무리마다 가장 많은 종의 수를 더하면:", ct.max(axis=1).sum())

정답을 한 번도 보지 않았는데도, 만들어진 세 무리가 실제 세 종과 거의 겹칩니다. 다만 모델은 각 무리가 '어떤 종'인지는 모릅니다. 이름을 붙이는 것은 사람의 몫입니다.

---
## 실습 문제

**문제 1.** `test_size`를 0.25에서 0.5로 바꿔(학습 데이터를 절반으로 줄여) 다시 나누고, k=1과 k=5의 학습 정확도와 테스트 정확도를 비교해 보세요. 결과가 어떻게 달라지나요?

In [ ]:
# 문제 1 풀이


**문제 2.** 특징을 부리 길이와 날개 길이 대신 **부리 길이와 부리 깊이**(`bill_length_mm`, `bill_depth_mm`)로 바꿔 산점도를 그리고, k=5 모델의 테스트 정확도를 구해 보세요. (`random_state=42`, `test_size=0.25` 그대로)

In [ ]:
# 문제 2 풀이


**문제 3.** 날개 길이 대신 **부리 길이**로 몸무게를 예측하는 규칙을 찾아 보세요. 두 경우의 산점도를 나란히 그려 보고, 어느 특징이 몸무게와 더 관계가 깊어 보이는지 설명하세요.

In [ ]:
# 문제 3 풀이


**문제 4.** 다음 문제는 지도 학습(분류·회귀), 비지도 학습, 강화 학습 가운데 무엇에 가까운지 답하고 이유를 쓰세요.
- (a) 쇼핑몰 고객을 구매 기록만 보고 비슷한 무리로 나누기
- (b) 아파트의 면적·층수·위치로 매매 가격 예측하기
- (c) 로봇이 넘어지지 않고 걷는 방법을 시행착오로 익히기
- (d) 사진을 보고 고양이·강아지·토끼 가운데 하나로 구별하기

(답을 이 칸 아래에 적으세요.)